# AI Hands-On — Week 13 Lab
## Quantization: From Scratch

**NTUA SEMFE · Postgraduate Course**

In this lab you will implement the core ideas from today's lecture completely from scratch using only NumPy. No quantization libraries. No `bitsandbytes`. No shortcuts.

The lab is split into three parts:

| Part | Topic | Core skill |
|------|-------|------------|
| 1 | RTN Quantization and the Outlier Problem | Understand *why* naive rounding fails |
| 2 | Hadamard Rotation | Verify the 5× error improvement from scratch |
| 3 | GPTQ-Style Optimization | Implement second-order error compensation |

Each **TODO** block tells you exactly what to implement. The **Expected output** line tells you what you should see when it works correctly. Do not modify pre-written cells.

> **Submission**: complete notebook with all TODO cells implemented and all output cells populated.


## Setup — Run this first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print("Setup complete.")


---
# Part 1 — RTN Quantization and the Outlier Problem

## Background

Symmetric per-tensor INT$b$ quantization maps a floating-point matrix $W$ to a discrete grid:

$$\hat{W} = s \cdot \operatorname{clip}\!\left(\operatorname{round}\!\left(\frac{W}{s}\right),\, -2^{b-1},\, 2^{b-1}-1\right)$$

The scale is determined by the largest absolute value in the tensor:

$$s = \frac{\max|W|}{2^{b-1}-1}$$

The problem: one outlier makes $s$ very large, crushing all small values to zero.


### Data — Weight matrix from the lecture (Example 1)

In [ ]:
# Weight matrix W with one outlier at position (3, 3)
W = np.array([
    [ 0.3989,  0.5305,  0.0145,  0.0387],
    [ 0.0005,  0.0359, -0.0435, -0.0252],
    [-0.0434,  0.0146,  0.0367, -0.0472],
    [ 0.0102, -0.0107,  0.0200, -6.6295],  # <-- outlier
])

x = np.array([1.0, -1.0, 0.5, 2.0])   # input vector
h = W @ x                               # true output

print("Weight matrix W:")
print(W)
print(f"\nInput x: {x}")
print(f"True output h: {h}")
print(f"\nMax |W|  = {np.max(np.abs(W)):.4f}  (the outlier)")
print(f"Max |W without outlier| ≈ {np.max(np.abs(W[:3, :3])):.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
norm = TwoSlopeNorm(vmin=-7, vcenter=0, vmax=0.6)
im = ax.imshow(W, cmap='RdBu_r', norm=norm)
plt.colorbar(im, ax=ax, shrink=0.85)
ax.set_title("Weight matrix W\n(outlier at [3,3] = −6.63)", fontsize=11)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{W[i,j]:.2f}", ha='center', va='center',
                fontsize=7.5, color='white' if abs(W[i,j]) > 1 else 'black')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
plt.tight_layout(); plt.show()


---
### TODO 1 — Implement `quantize_rtn(W, bits)`

Implement symmetric per-tensor Round-to-Nearest quantization.

**Steps:**
1. Compute scale: `s = max(|W|) / (2^(bits-1) - 1)`
2. Integer map: `q = clip(round(W / s), -2^(bits-1), 2^(bits-1) - 1)`
3. Dequantize: `W_hat = s * q`
4. Return `W_hat`

> **Expected output (bits=4):** `Scale s ≈ 0.9471`


In [ ]:
def quantize_rtn(W, bits=4):
    """
    Symmetric per-tensor Round-to-Nearest quantization.
    Returns dequantized matrix W_hat (same shape as W, float64).
    """
    # TODO: implement the three steps above
    # Hint: 2**(bits-1) - 1 gives the maximum integer level (= 7 for 4-bit)
    pass


# ── Quick sanity check ──
if quantize_rtn is not None and quantize_rtn(W) is not None:
    s = np.max(np.abs(W)) / (2**(4-1) - 1)
    print(f"Scale s = {s:.4f}  (expected ≈ 0.9471)")


---
### TODO 2 — Apply RTN to W and measure output error

1. Call `quantize_rtn(W, bits=4)` and store the result as `W_hat_rtn`
2. Compute the quantized output: `h_hat_rtn = W_hat_rtn @ x`
3. Compute and print the Euclidean error: `np.linalg.norm(h_hat_rtn - h)`

> **Expected output:** `RTN error ≈ 0.916`


In [ ]:
# TODO: apply RTN and compute error
W_hat_rtn = None   # replace with your call
h_hat_rtn  = None  # replace with W_hat_rtn @ x

if W_hat_rtn is not None and h_hat_rtn is not None:
    error_rtn = np.linalg.norm(h_hat_rtn - h)
    print("Quantized W (INT4, naive RTN):")
    print(W_hat_rtn)
    print(f"\nQuantized output h_hat: {h_hat_rtn}")
    print(f"True output h:           {h}")
    print(f"\nRTN error: {error_rtn:.4f}  (expected ≈ 0.916)")


In [ ]:
# Pre-written visualisation — run after completing TODO 2
if W_hat_rtn is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
    norm = TwoSlopeNorm(vmin=-7, vcenter=0, vmax=0.6)

    for ax, mat, title in zip(axes, [W, W_hat_rtn], ["Original W", "After naive INT4"]):
        im = ax.imshow(mat, cmap='RdBu_r', norm=norm)
        ax.set_title(title, fontsize=11)
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{mat[i,j]:.2f}", ha='center', va='center',
                        fontsize=8, color='white' if abs(mat[i,j]) > 1 else 'black')
        ax.set_xticks(range(4)); ax.set_yticks(range(4))

    plt.suptitle("The outlier at [3,3] forces scale → most entries become 0", fontsize=10)
    plt.tight_layout(); plt.show()


---
### TODO 3 — Implement SVD Separation

Implement `svd_separate(W, k)` that:
1. Computes the full SVD of W: `U, s, Vt = np.linalg.svd(W, full_matrices=True)`
2. Builds the rank-$k$ outlier matrix: $W_\text{outlier} = U_k \Sigma_k V_k^T$
3. Builds the clean residual: $W_\text{clean} = W - W_\text{outlier}$
4. Returns `(W_outlier, W_clean)`

Then:
- Quantize `W_clean` with `quantize_rtn(W_clean, bits=4)` and store as `W_clean_q`
- Compute the **combined** output for k=1 and k=2:
  `h_hat = quantize_rtn(W_clean) @ x + W_outlier @ x`

> **Expected errors:** Rank-1 ≈ 0.049 · Rank-2 ≈ 0.003


In [ ]:
def svd_separate(W, k):
    """
    Split W into a rank-k outlier component and a clean residual.
    Returns: (W_outlier, W_clean) both of shape W.shape
    """
    # TODO: implement SVD separation
    # Hint: np.linalg.svd returns (U, singular_values, Vt)
    # To build rank-k: U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    pass


# ── Verify for k = 1 and k = 2 ──
errors_svd = {}
for k in [1, 2]:
    result = svd_separate(W, k)
    if result is not None:
        W_outlier, W_clean = result
        W_clean_q = quantize_rtn(W_clean, bits=4)
        if W_clean_q is not None:
            h_hat = W_clean_q @ x + W_outlier @ x
            err = np.linalg.norm(h_hat - h)
            errors_svd[k] = err
            print(f"SVD rank-{k} error: {err:.4f}  (expected: rank-1 ≈ 0.049, rank-2 ≈ 0.003)")


In [ ]:
# Pre-written: error vs rank — run after completing TODO 3
if errors_svd:
    ks = list(range(0, 5))
    errs = []
    for k in ks:
        if k == 0:
            errs.append(np.linalg.norm(quantize_rtn(W, bits=4) @ x - h)
                        if quantize_rtn(W) is not None else np.nan)
        else:
            res = svd_separate(W, k)
            if res is not None:
                Wo, Wc = res
                Wcq = quantize_rtn(Wc, bits=4)
                if Wcq is not None:
                    errs.append(np.linalg.norm((Wcq @ x + Wo @ x) - h))
                else:
                    errs.append(np.nan)
            else:
                errs.append(np.nan)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    valid = [(k, e) for k, e in zip(ks, errs) if not np.isnan(e)]
    if valid:
        ks_v, errs_v = zip(*valid)
        bars = ax.bar(ks_v, errs_v, color=['#c0392b'] + ['#2980b9']*4, width=0.6, zorder=3)
        ax.axhline(errs[0] if not np.isnan(errs[0]) else 0.916,
                   color='#c0392b', linestyle='--', lw=1.2, label='Naive RTN')
        for bar, val in zip(bars, errs_v):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        ax.set_xlabel("SVD rank k")
        ax.set_ylabel("Output error ‖ĥ − h‖")
        ax.set_title("Reconstruction error vs SVD rank\n(INT4 on clean part + full-precision outlier)")
        ax.set_xticks(ks_v)
        ax.set_xticklabels(['RTN (k=0)'] + [f'k={k}' for k in ks_v[1:]])
        ax.grid(axis='y', alpha=0.4); ax.legend()
        plt.tight_layout(); plt.show()


---
### 🔵 Extension 1

Repeat the SVD separation experiment on a **random** matrix with no planted outlier:
```python
np.random.seed(0)
W_random = np.random.randn(4, 4) * 0.4
```
Plot the error vs rank curve. Does SVD separation still help? Why or why not?


In [ ]:
# Extension 1 — your code here


---
# Part 2 — Hadamard Rotation

## Background

The Hadamard matrix $H_n$ ($n = 2^d$) satisfies $H_n H_n^T = I_n$. This lets us rewrite any matrix-vector product without changing it:

$$y = Wx = W(HH^T)x = \underbrace{(WH)}_{W'}\underbrace{(H^Tx)}_{x'} = W'x'$$

After rotation, the outlier is spread across all columns of $W'$, reducing $\max|W'|$ by a factor $\approx \sqrt{n}$. A smaller max value → smaller scale $s$ → finer quantization grid → less rounding error.


### Pre-written: `build_hadamard(n)`

In [ ]:
def build_hadamard(n):
    """
    Build normalized n×n Hadamard matrix (entries ±1/√n).
    H @ H.T = I  by construction.
    n must be a power of 2.
    """
    assert n > 0 and (n & (n - 1)) == 0, "n must be a power of 2"
    if n == 1:
        return np.array([[1.0]])
    H_half = build_hadamard(n // 2)
    H = np.block([[H_half,  H_half],
                  [H_half, -H_half]])
    return H / np.sqrt(2)   # normalize so H @ H.T = I

H4 = build_hadamard(4)
print("4×4 normalized Hadamard matrix H:")
print(H4)
print(f"\nAll entries are ±{1/2:.4f}  (= ±1/√4)")


---
### TODO 4 — Verify that H is orthonormal

1. Compute `H4 @ H4.T` and print the result.
2. Compute `np.max(np.abs(H4 @ H4.T - np.eye(4)))` — this should be ≈ 0.
3. Repeat for `n = 256` and `n = 1024`.

> **Expected:** max deviation from identity < 1e-13 for all sizes.


In [ ]:
# TODO: verify orthonormality for n = 4, 256, 1024
for n in [4, 256, 1024]:
    H = build_hadamard(n)
    # compute H @ H.T and the max deviation from identity
    pass


---
### TODO 5 — Apply the Hadamard rotation and measure the scale reduction

Using the same `W` and `x` from Part 1:
1. Compute `W_prime = W @ H4`
2. Compute `x_prime = H4.T @ x`
3. Print `max(|W|)` and `max(|W_prime|)`.
4. Compute and print the ratio. It should be close to $\sqrt{4} = 2$.
5. Compute `s_naive` and `s_rotated` (the INT4 scales before and after rotation).

> **Expected:** `max|W'| ≈ 3.31`, ratio ≈ **2.0×**, `s_rotated ≈ 0.475` (vs `s_naive ≈ 0.947`).


In [ ]:
# TODO: rotate W and measure scale reduction
W_prime = None  # W @ H4
x_prime = None  # H4.T @ x

if W_prime is not None:
    max_W       = np.max(np.abs(W))
    max_W_prime = np.max(np.abs(W_prime))

    # TODO: print the comparison and the ratio
    pass


In [ ]:
# Pre-written: histogram comparison — run after TODO 5
if W_prime is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    for ax, mat, title, color in zip(
        axes, [W, W_prime],
        ["Original W\n(one large outlier)", "Rotated W' = WH\n(energy spread uniformly)"],
        ['#c0392b', '#2980b9']
    ):
        ax.hist(mat.flatten(), bins=20, color=color, alpha=0.8, edgecolor='white')
        ax.axvline(0, color='black', lw=0.8, linestyle='--')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Weight value")
    axes[0].set_ylabel("Count")
    plt.suptitle("Hadamard rotation transforms the weight distribution", fontsize=11)
    plt.tight_layout(); plt.show()


---
### TODO 6 — Quantize the rotated matrix and print the full comparison table

1. Quantize `W_prime` with `quantize_rtn(W_prime, bits=4)` → `W_prime_hat`
2. Compute the Hadamard + INT4 output: `h_hat_had = W_prime_hat @ x_prime`
3. Compute error: `np.linalg.norm(h_hat_had - h)`
4. Print the complete error comparison table for all methods.

> **Expected error (Hadamard + INT4) ≈ 0.21**


In [ ]:
# TODO: quantize W' and compute the output error
W_prime_hat = None   # quantize_rtn(W_prime, bits=4)
h_hat_had   = None   # W_prime_hat @ x_prime

if h_hat_had is not None:
    error_had = np.linalg.norm(h_hat_had - h)

    # TODO: build and print the comparison table
    # Include: Naive INT4, Hadamard+INT4, SVD Rank-1, SVD Rank-2
    # Format: method name, error value, improvement vs naive RTN
    pass


In [ ]:
# Pre-written: comparison bar chart — run after TODO 6
methods, values, colors = [], [], []

if 'error_rtn' in dir() or 'error_rtn' in locals() or True:
    try:
        e_rtn = np.linalg.norm(quantize_rtn(W, 4) @ x - h)
        methods.append('Naive\nINT4'); values.append(e_rtn); colors.append('#c0392b')
    except: pass

if h_hat_had is not None:
    error_had = np.linalg.norm(h_hat_had - h)
    methods.append('Hadamard\n+ INT4'); values.append(error_had); colors.append('#2980b9')

for k, col in [(1, '#27ae60'), (2, '#16a085')]:
    res = svd_separate(W, k) if svd_separate(W, 1) is not None else None
    if res is not None:
        Wo, Wc = res
        Wcq = quantize_rtn(Wc, 4)
        if Wcq is not None:
            err = np.linalg.norm((Wcq @ x + Wo @ x) - h)
            methods.append(f'SVD\nRank-{k}'); values.append(err); colors.append(col)

if methods:
    fig, ax = plt.subplots(figsize=(7, 3.8))
    bars = ax.bar(methods, values, color=colors, width=0.55, zorder=3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylabel("Output error ‖ĥ − h‖")
    ax.set_title("All methods — same 4×4 matrix, same x", fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()


---
### TODO 7 — QuIP# style: add random sign matrices

In QuIP#, the rotation is $W' = W D_2 H$ where $D_2$ is a diagonal matrix of random $\pm 1$ entries. At inference: $y = W' \cdot (H^T D_2 x)$.

1. Generate a random $\pm 1$ diagonal: `d2 = rng.choice([-1, 1], size=4)`
2. Build `D2 = np.diag(d2)`
3. Compute `W_quip = W @ D2 @ H4` and `x_quip = H4.T @ D2 @ x`
4. Quantize `W_quip` and compute the output error.
5. Run 50 random seeds. Print `mean ± std` of the error.

> **Goal:** see whether random signs reduce the mean error compared to plain Hadamard (≈ 0.185).


In [ ]:
rng = np.random.default_rng(42)

# TODO: single run
d2 = rng.choice([-1, 1], size=4)
D2 = np.diag(d2)

W_quip = None   # W @ D2 @ H4
x_quip = None   # H4.T @ D2 @ x

if W_quip is not None:
    W_quip_hat = quantize_rtn(W_quip, bits=4)
    if W_quip_hat is not None:
        h_quip = W_quip_hat @ x_quip
        print(f"Single run error: {np.linalg.norm(h_quip - h):.4f}")

# TODO: 50 random seeds — print mean ± std
errors_quip = []
for seed in range(50):
    pass  # your loop here

if errors_quip:
    print(f"\n50 random seeds: mean = {np.mean(errors_quip):.4f}  std = {np.std(errors_quip):.4f}")
    print(f"Plain Hadamard:  ≈ 0.21")


---
### 🔵 Extension 2

Repeat the QuIP# experiment with `n = 64` using a randomly generated $64 \times 64$ weight matrix with 3 planted outlier columns. Do the random signs provide a more consistent benefit at larger $n$?


In [ ]:
# Extension 2 — your code here


---
# Part 3 — GPTQ-Style Optimization

## Background

Instead of rounding each weight independently (RTN), GPTQ formulates PTQ as a layer-wise least-squares problem:

$$\min_{W_q \in \mathcal{Q}}\; \|WX - W_q X\|_F^2$$

where $X \in \mathbb{R}^{d_{in} \times N}$ is a calibration activation matrix. GPTQ quantizes column by column. After quantizing column $j$, it propagates the rounding error to the remaining columns using the inverse Hessian $H^{-1}$:

$$W[:, j+1:] \;\mathrel{-}=\; \frac{e_j}{[H^{-1}]_{jj}} \cdot H^{-1}_{j,\,j+1:}$$

where $e_j = \hat{w}_j - w_j$ is the quantization error for column $j$, and $H = 2XX^T$.

**Intuition**: rounding column $j$ introduces an error that shifts the output. The Hessian tells us which adjustments to the remaining columns will cancel that shift.


### Data — 8×8 weight matrix with calibration activations

In [ ]:
rng3 = np.random.default_rng(7)

# 8×8 weight matrix with planted outliers in two columns
W8 = rng3.standard_normal((8, 8)) * 0.35
W8[:, 2]  = rng3.standard_normal(8) * 7.5    # outlier column
W8[4, 2]  = 11.2                              # extra large entry
W8[:, 6]  = rng3.standard_normal(8) * 4.0    # second outlier column

# Calibration activations: 8 input features, 24 samples
X8 = rng3.standard_normal((8, 24))

# True output on calibration data
Y8_true = W8 @ X8

def recon_error(W_q, W, X):
    """Reconstruction error ||W_q X - W X||_F"""
    return np.linalg.norm(W_q @ X - W @ X, 'fro')

print("W8 shape:", W8.shape)
print("X8 shape:", X8.shape)
print(f"Max |W8| = {np.max(np.abs(W8)):.3f}  (in the outlier column)")
print(f"\nBaseline (naive per-tensor RTN) error: ", end="")
W8_rtn = quantize_rtn(W8, bits=4)
if W8_rtn is not None:
    print(f"{recon_error(W8_rtn, W8, X8):.4f}")
else:
    print("implement quantize_rtn first")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
norm8 = TwoSlopeNorm(vmin=W8.min(), vcenter=0, vmax=W8.max())
im = ax.imshow(W8, cmap='RdBu_r', norm=norm8)
plt.colorbar(im, ax=ax)
ax.set_title("8×8 Weight matrix W8\n(two outlier columns: 2 and 6)", fontsize=11)
ax.set_xlabel("Column (input feature)")
ax.set_ylabel("Row (output neuron)")
plt.tight_layout(); plt.show()


---
### TODO 8 — Column-by-column RTN (no error compensation)

Implement `quantize_columns_no_comp(W, bits)`:
1. Create `W_q = W.copy()`
2. For each column `j` from 0 to `W.shape[1] - 1`:
   - Compute the per-column scale: `s_j = max(|W_q[:, j]|) / (2^(bits-1) - 1)`
   - Round the column: `W_q[:, j] = s_j * clip(round(W_q[:, j] / s_j), ...)`
3. After each column quantization, record the current reconstruction error `recon_error(W_q, W8, X8)`
4. Return `(W_q, list_of_errors)`

> **Expected:** final error should be lower than naive per-tensor RTN (which has one global scale dominated by the outlier).


In [ ]:
def quantize_columns_no_comp(W, bits=4):
    """
    Column-by-column per-column RTN — no error compensation.
    Returns (W_q, errors) where errors[j] = recon error after quantizing column j.
    """
    W_q = W.copy()
    errors = []
    levels = 2**(bits-1) - 1   # = 7 for 4-bit

    for j in range(W.shape[1]):
        # TODO: compute per-column scale and quantize column j
        # then append recon_error(W_q, W8, X8) to errors
        pass

    return W_q, errors


W8_col, errors_col = quantize_columns_no_comp(W8, bits=4)

if errors_col:
    print(f"Per-column RTN final error: {errors_col[-1]:.4f}")
    print(f"Naive per-tensor RTN error: ", end="")
    if W8_rtn is not None:
        print(f"{recon_error(W8_rtn, W8, X8):.4f}")
    print("\nError after each column quantization:")
    for j, e in enumerate(errors_col):
        print(f"  col {j}: {e:.4f}")


---
### TODO 9 — GPTQ with Hessian-based error compensation

Implement `gptq_quantize(W, X, bits)`:
1. Compute the Hessian: `H = 2 * X @ X.T`  *(shape: in_features × in_features)*
2. Compute the inverse with small regularization: `H_inv = np.linalg.inv(H + 1e-5 * np.eye(d))`
3. Create `W_q = W.copy()`
4. For each column `j`:
   a. Compute per-column scale `s_j` and quantize column `j`
   b. Compute the rounding error vector: `e_j = W_q[:, j] - W[:, j]` *(shape: out_features,)*
   c. Store `W_q[:, j]` (quantized)
   d. Compensate the remaining columns:
      ```
      W_q[:, j+1:] -= np.outer(e_j, H_inv[j, j+1:]) / H_inv[j, j]
      ```
   e. Record `recon_error(W_q, W8, X8)`
5. Return `(W_q, errors)`

> **Expected:** GPTQ error should be noticeably lower than per-column RTN without compensation.


In [ ]:
def gptq_quantize(W, X, bits=4):
    """
    GPTQ: column-by-column quantization with second-order error compensation.
    Returns (W_q, errors) where errors[j] = recon error after quantizing column j.
    """
    d_in = W.shape[1]
    levels = 2**(bits-1) - 1

    # TODO step 1: compute Hessian H = 2 * X @ X.T
    H = None

    # TODO step 2: compute H_inv with regularization
    H_inv = None

    if H_inv is None:
        return None, []

    W_q = W.copy()
    errors = []

    for j in range(d_in):
        # TODO: quantize column j (per-column scale)
        # TODO: compute e_j = W_q[:, j] - W[:, j]   (BEFORE storing the quantized value)
        # TODO: store quantized column in W_q[:, j]
        # TODO: compensate remaining columns (only if j < d_in - 1)
        # TODO: append error
        pass

    return W_q, errors


W8_gptq, errors_gptq = gptq_quantize(W8, X8, bits=4)

if errors_gptq:
    print(f"GPTQ final error:          {errors_gptq[-1]:.4f}")
    if errors_col:
        print(f"Per-column RTN error:      {errors_col[-1]:.4f}")
    if W8_rtn is not None:
        print(f"Naive per-tensor RTN:      {recon_error(W8_rtn, W8, X8):.4f}")


---
### TODO 10 — Plot the error trajectory for both methods

Plot `errors_col` and `errors_gptq` on the same axes (error vs column index).

**What to show:**
- x-axis: column index (0–7)
- y-axis: reconstruction error `||W_q X - WX||_F`
- Two lines: per-column RTN (no compensation) and GPTQ (with compensation)
- Mark the two outlier columns (2 and 6) with vertical dashed lines

**Observation to answer in a markdown cell below**: at which column does GPTQ show the largest improvement? Why does it happen at that column specifically?


In [ ]:
# TODO: plot error trajectories
# errors_col   — from quantize_columns_no_comp
# errors_gptq  — from gptq_quantize
# Hint: plt.plot(range(8), errors_col, ...) etc.

if errors_col and errors_gptq:
    pass  # your plotting code here


**Your observation here** (double-click to edit):

> *At which column does the compensation provide the biggest benefit, and why?*


In [ ]:
# Pre-written: final summary bar chart — run after all TODOs
summary = {}
if quantize_rtn(W8) is not None:
    summary['Naive RTN\n(per-tensor)'] = recon_error(quantize_rtn(W8, 4), W8, X8)
if errors_col:
    summary['Per-column\nRTN'] = errors_col[-1]
if errors_gptq:
    summary['GPTQ\n(+compensation)'] = errors_gptq[-1]

if summary:
    fig, ax = plt.subplots(figsize=(7, 3.8))
    color_map = {'Naive RTN\n(per-tensor)': '#c0392b',
                 'Per-column\nRTN': '#e67e22',
                 'GPTQ\n(+compensation)': '#27ae60'}
    bars = ax.bar(summary.keys(), summary.values(),
                  color=[color_map.get(k, '#2980b9') for k in summary],
                  width=0.55, zorder=3)
    for bar, val in zip(bars, summary.values()):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel("Reconstruction error  ‖WqX − WX‖_F")
    ax.set_title("INT4 quantization — 8×8 matrix with outliers\nAll methods compared", fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()


---
### 🔵 Extension 3 — Does the order of columns matter for GPTQ?

GPTQ processes columns left-to-right. Some papers suggest sorting columns by their Hessian diagonal (most important first vs least important first) before quantizing.

1. Sort columns by `H_inv[j, j]` in ascending order (most sensitive last).
2. Run GPTQ on the reordered matrix.
3. Compare final reconstruction error with the standard order.

*Hint*: use `np.argsort` and reorder both `W` and `H` consistently.


In [ ]:
# Extension 3 — your code here


---
# Summary

You have now implemented from scratch the three foundational building blocks of modern LLM quantization:

| What you built | What it corresponds to in the literature |
|---|---|
| `quantize_rtn` | Baseline used in almost all quantization papers |
| `svd_separate` | Core of SVDQuant and SVD-LLM |
| Hadamard rotation + RTN | Core of QuIP# (Step 1 + Step 2) |
| Random sign matrices | The $D_1, D_2$ matrices in QuIP# |
| `gptq_quantize` | GPTQ (Frantar et al., 2022) — the production standard |

The methods you implemented here run on the world's largest language models. The mathematics is the same — the engineering challenge is making them work at the scale of billions of parameters and thousands of GPU hours.

---
*AI Hands-On · Week 13 · NTUA SEMFE*
